# 6. Hypothesis Testing

**Statistical Foundations for Data Science — Notebook 6 of 8**

You changed the checkout button from blue to green and conversions rose from 4.1% to 4.6%.
Is the button better, or did you just get a lucky week? **Hypothesis testing** is the formal
machinery for answering that class of question: *could this pattern plausibly have arisen
from chance alone?*

This notebook builds the framework. Notebooks 7 and 8 apply it to the two most common
cases (t-tests for means, chi-square for categories).

### What you will learn

1. Null and alternative hypotheses, and how to state them correctly
2. **Test statistics**, **p-values**, and **significance levels**
3. **Type I** and **Type II** errors; **power**; and how to trade them off
4. One-tailed vs. two-tailed tests
5. **Critical value** and **p-value** approaches (they always agree)
6. **Permutation tests** — hypothesis testing from first principles, no formulas
7. **Effect size** and why significance is not importance
8. **Multiple comparisons**: p-hacking, Bonferroni, and FDR
9. A decision guide for choosing the right test

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(seed=99)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

---
## 6.1 The logic of a hypothesis test

Hypothesis testing is **proof by contradiction with probabilities**:

1. Assume nothing interesting is happening (the **null hypothesis** $H_0$)
2. Work out what the data *should* look like under that assumption
3. Compare your actual data to that expectation
4. If your data would be very surprising under $H_0$, reject $H_0$

### Stating the hypotheses

| | Symbol | Meaning | Contains |
|---|---|---|---|
| **Null** | $H_0$ | The boring/default claim — no effect, no difference | always an equality: $=$, $\le$, $\ge$ |
| **Alternative** | $H_1$ or $H_a$ | What you suspect | strict inequality: $\ne$, $<$, $>$ |

Rules that matter:

- The two must be **mutually exclusive** and cover all possibilities
- $H_0$ always gets the equals sign, because you need a specific value to compute with
- State them **before** looking at the data. Choosing $H_1$ after seeing the direction of
  your result doubles your false-positive rate.

### The asymmetry you must internalise

You can **reject** $H_0$, or you can **fail to reject** $H_0$. You can never *accept* or
*prove* $H_0$. "No significant difference" means "we did not find enough evidence", which
is not the same as "there is no difference" — you may simply have had too little data.

---
## 6.2 Test statistic, null distribution, p-value

A **test statistic** compresses your data into a single number measuring how far it sits
from what $H_0$ predicts. Most look like:

$$\text{test statistic} = \frac{\text{observed} - \text{expected under } H_0}{\text{standard error}}$$

The **null distribution** is the sampling distribution of that statistic *if $H_0$ were
true*. The **p-value** is the probability, under $H_0$, of getting a result at least as
extreme as what you observed.

$$p = P(\text{statistic at least as extreme as observed} \mid H_0 \text{ true})$$

### What a p-value is *not*

| ❌ Wrong | ✅ Right |
|---|---|
| The probability $H_0$ is true | The probability of this data *given* $H_0$ |
| The probability your result was a fluke | How surprising the data is under $H_0$ |
| A measure of effect size | Says nothing about magnitude |
| $p = 0.049$ true, $p = 0.051$ false | The threshold is a convention, not a law of nature |

Formally: $p = P(\text{data} \mid H_0)$, **not** $P(H_0 \mid \text{data})$. That is the
same $P(A\mid B)$ vs $P(B\mid A)$ flip from Notebook 1.

In [ ]:
# A concrete example, computed from scratch.
# Claim: a machine fills bottles to 500 ml. Historical sd is known to be 8 ml.
# We sample 40 bottles and get a mean of 496.5 ml. Is the machine off-target?

mu_0, sigma, n = 500, 8, 40
x_bar = 496.5

se = sigma / np.sqrt(n)
z_stat = (x_bar - mu_0) / se
p_two_sided = 2 * stats.norm.sf(abs(z_stat))

print(f"H0: mu = {mu_0} ml      H1: mu != {mu_0} ml")
print(f"Standard error   = {sigma}/sqrt({n}) = {se:.4f}")
print(f"z statistic      = ({x_bar} - {mu_0}) / {se:.4f} = {z_stat:.4f}")
print(f"p-value          = {p_two_sided:.5f}")
print(f"\nInterpretation: if the machine were perfectly calibrated, we would see a")
print(f"sample mean this far from 500 about {p_two_sided*100:.2f}% of the time.")
print(f"Decision at alpha = 0.05: {'REJECT H0' if p_two_sided < 0.05 else 'fail to reject H0'}")

In [ ]:
# Visualise the null distribution, the observed statistic, and the p-value area
xs = np.linspace(-4, 4, 800)
pdf = stats.norm.pdf(xs)

plt.plot(xs, pdf, color="black", lw=1.6)
tail_hi = xs >= abs(z_stat)
tail_lo = xs <= -abs(z_stat)
plt.fill_between(xs[tail_hi], pdf[tail_hi], color="crimson", alpha=0.6)
plt.fill_between(xs[tail_lo], pdf[tail_lo], color="crimson", alpha=0.6,
                 label=f"p-value area = {p_two_sided:.4f}")
plt.axvline(z_stat, color="crimson", lw=2, ls="--", label=f"observed z = {z_stat:.2f}")
for cv in (-1.96, 1.96):
    plt.axvline(cv, color="steelblue", lw=1.4, ls=":")
plt.text(2.05, 0.30, "critical\nvalue\n1.96", fontsize=8, color="steelblue")
plt.title("Null distribution: how surprising is our sample?")
plt.xlabel("z"); plt.ylabel("density"); plt.legend(fontsize=8)
plt.show()

### The five steps of any hypothesis test

1. **State** $H_0$ and $H_1$, and choose $\alpha$ **before** seeing the data
2. **Check assumptions** (independence, distribution, sample size)
3. **Compute** the test statistic and the p-value
4. **Decide**: reject $H_0$ if $p < \alpha$
5. **Report in context**: effect size, confidence interval, and practical meaning

Step 5 is the one everybody skips, and it is the one that matters to your stakeholders.

---
## 6.3 Two ways to decide (they always agree)

**p-value approach:** reject $H_0$ if $p < \alpha$.

**Critical value approach:** reject $H_0$ if the test statistic falls in the **rejection
region**, i.e. beyond $\pm z_{\alpha/2}$ for a two-tailed test.

These are the same rule read from opposite ends of the distribution. The p-value approach
is preferred in modern practice because it reports *how* surprising the data is, rather
than just a binary verdict.

In [ ]:
alpha = 0.05
crit = stats.norm.ppf(1 - alpha / 2)

print(f"alpha = {alpha}, two-tailed critical values = +/-{crit:.4f}")
print(f"Observed z = {z_stat:.4f}")
print(f"|z| > critical value?  {abs(z_stat) > crit}  -> reject H0")
print(f"p < alpha?             {p_two_sided < alpha}  -> reject H0")
print("\nBoth approaches must always agree -- they are the same statement.")

print("\nCritical values you should recognise on sight (two-tailed):")
for a in (0.10, 0.05, 0.01, 0.001):
    print(f"  alpha = {a:<6} -> z = +/-{stats.norm.ppf(1 - a/2):.3f}")

---
## 6.4 One-tailed vs two-tailed tests

| | Two-tailed | One-tailed |
|---|---|---|
| $H_1$ | $\mu \ne \mu_0$ | $\mu > \mu_0$ (or $\mu < \mu_0$) |
| Question | "Is it different?" | "Is it better?" |
| Rejection region | Both tails, $\alpha/2$ each | One tail, all of $\alpha$ |
| Power | Lower | Higher **in the specified direction** |
| Risk | — | **Blind** to an effect in the other direction |

Use a one-tailed test only when a change in the opposite direction would be **irrelevant to
your decision**, and commit to it in advance. Switching to one-tailed after seeing which way
the data went is a form of p-hacking that silently doubles your false-positive rate.

In [ ]:
z = z_stat
p_two   = 2 * stats.norm.sf(abs(z))
p_left  = stats.norm.cdf(z)          # H1: mu < 500
p_right = stats.norm.sf(z)           # H1: mu > 500

print(f"Observed z = {z:.4f}\n")
print(f"Two-tailed  (H1: mu != 500) : p = {p_two:.5f}")
print(f"Left-tailed (H1: mu <  500) : p = {p_left:.5f}   <- exactly half of two-tailed")
print(f"Right-tailed(H1: mu >  500) : p = {p_right:.5f}   <- nearly 1: wrong direction")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
xs = np.linspace(-4, 4, 600); pdf = stats.norm.pdf(xs)
regions = [("two-tailed", np.abs(xs) >= 1.96), ("left-tailed", xs <= -1.645),
           ("right-tailed", xs >= 1.645)]
for ax, (name, mask) in zip(axes, regions):
    ax.plot(xs, pdf, color="black")
    ax.fill_between(xs[mask], pdf[mask], color="crimson", alpha=0.6)
    ax.axvline(z, color="steelblue", lw=2, ls="--")
    ax.set_title(f"{name} rejection region (alpha=0.05)", fontsize=10)
    ax.set_yticks([])
plt.tight_layout(); plt.show()

---
## 6.5 Type I and Type II errors

Every test can be wrong in two ways:

| | **$H_0$ is actually true** | **$H_0$ is actually false** |
|---|---|---|
| **Reject $H_0$** | ❌ **Type I error** (false positive), probability $\alpha$ | ✅ Correct — a true detection, probability $1-\beta$ = **power** |
| **Fail to reject** | ✅ Correct, probability $1-\alpha$ | ❌ **Type II error** (false negative), probability $\beta$ |

- **$\alpha$** (significance level) — the false-positive rate you are willing to accept.
  Conventionally 0.05, meaning you accept crying wolf 1 time in 20.
- **$\beta$** — the miss rate. **Power** $= 1 - \beta$ is the probability of detecting a
  real effect. 0.80 is the usual minimum target.

**Analogy.** A medical test: Type I error = telling a healthy person they are ill; Type II
error = missing a real illness. Which is worse depends entirely on the context, and that is
a *domain* decision, not a statistical one.

**The trade-off:** lowering $\alpha$ (being more cautious) always raises $\beta$ for a fixed
sample size. The only way to reduce both is **more data**.

In [ ]:
# Demonstrate the Type I error rate: run 10,000 tests where H0 is TRUE.
alpha = 0.05
reps = 10_000
n = 30

p_values = []
for _ in range(reps):
    s = rng.normal(loc=100, scale=15, size=n)        # H0 (mu = 100) is genuinely true
    p_values.append(stats.ttest_1samp(s, popmean=100).pvalue)
p_values = np.array(p_values)

false_positives = (p_values < alpha).mean()
print(f"H0 is TRUE in all {reps:,} tests.")
print(f"Rejected H0 in {(p_values < alpha).sum():,} tests = {false_positives:.4f}")
print(f"That matches alpha = {alpha} exactly. Type I errors are not a bug --")
print("they are the price of admission, and you set the price yourself.")

plt.hist(p_values, bins=40, color="steelblue", edgecolor="white")
plt.axvline(alpha, color="crimson", lw=2, ls="--", label="alpha = 0.05")
plt.xlabel("p-value"); plt.ylabel("frequency")
plt.title("Under a true H0, p-values are UNIFORM on [0,1]")
plt.legend(); plt.show()

In [ ]:
# Type II error and power: now H0 is FALSE (true mean is 105, not 100)
true_mu, n = 105, 30
p_values_false = np.array([
    stats.ttest_1samp(rng.normal(true_mu, 15, n), popmean=100).pvalue for _ in range(reps)
])

power_emp = (p_values_false < alpha).mean()
print(f"True mean is {true_mu}, so H0 (mu=100) is FALSE.")
print(f"Detected the effect in {power_emp:.4f} of tests  ->  power = {power_emp:.3f}")
print(f"Missed it in {1 - power_emp:.4f} of tests        ->  beta = {1 - power_emp:.3f}")
print(f"\nWith n = {n} we miss a real 5-point effect about {(1-power_emp)*100:.0f}% of the time.")

In [ ]:
# The picture everybody should see once: two overlapping distributions,
# with alpha, beta and power marked.
mu0, mu1, se_ = 0, 2.2, 1.0
crit = stats.norm.ppf(1 - 0.05)          # one-tailed for clarity
xs = np.linspace(-4, 7, 900)

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.plot(xs, stats.norm(mu0, se_).pdf(xs), color="steelblue", lw=2, label="distribution if H0 true")
ax.plot(xs, stats.norm(mu1, se_).pdf(xs), color="seagreen",  lw=2, label="distribution if H1 true")

m = xs >= crit
ax.fill_between(xs[m], stats.norm(mu0, se_).pdf(xs[m]), color="crimson", alpha=0.55,
                label="alpha: Type I error")
m2 = xs <= crit
ax.fill_between(xs[m2], stats.norm(mu1, se_).pdf(xs[m2]), color="orange", alpha=0.45,
                label="beta: Type II error")
ax.fill_between(xs[m], stats.norm(mu1, se_).pdf(xs[m]), color="seagreen", alpha=0.25,
                label="power = 1 - beta")
ax.axvline(crit, color="black", lw=1.8, ls="--")
ax.text(crit + 0.05, 0.42, "critical value", fontsize=9)
ax.set_xlabel("test statistic"); ax.set_ylabel("density")
ax.set_title("alpha, beta and power on one picture")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

print(f"alpha = {stats.norm(mu0, se_).sf(crit):.3f}")
print(f"beta  = {stats.norm(mu1, se_).cdf(crit):.3f}")
print(f"power = {stats.norm(mu1, se_).sf(crit):.3f}")
print("\nMove the dashed line right: alpha shrinks, beta grows. There is no free lunch --")
print("only a bigger sample separates the two curves.")

### Power analysis: how big a sample do you need?

Power depends on four quantities; fix any three and the fourth is determined:

- **Effect size** — how big the real difference is (bigger = easier to detect)
- **Sample size $n$** — more data = more power
- **$\alpha$** — a stricter threshold costs power
- **Variability $\sigma$** — noisier data = less power

**Always do this before collecting data.** An underpowered study wastes the effort: it
cannot detect the effect it was built to find, and if it does report significance the
estimate is usually badly inflated.

In [ ]:
def power_one_sample(effect_size, n, alpha=0.05, two_sided=True):
    '''Power of a one-sample z/t test. effect_size = (mu1 - mu0)/sigma (Cohen's d).'''
    crit = stats.norm.ppf(1 - alpha / (2 if two_sided else 1))
    ncp = effect_size * np.sqrt(n)                 # non-centrality
    return stats.norm.sf(crit - ncp) + (stats.norm.cdf(-crit - ncp) if two_sided else 0.0)

ns = np.arange(5, 401)
for d, label in [(0.2, "small (d=0.2)"), (0.5, "medium (d=0.5)"), (0.8, "large (d=0.8)")]:
    plt.plot(ns, [power_one_sample(d, m) for m in ns], lw=2, label=label)
plt.axhline(0.80, color="crimson", ls="--", label="conventional target 0.80")
plt.xlabel("sample size n"); plt.ylabel("power")
plt.title("Power curves: small effects need big samples")
plt.legend(); plt.ylim(0, 1.02)
plt.show()

print("Sample size needed for 80% power (two-sided, alpha=0.05):")
for d in (0.2, 0.5, 0.8, 1.0):
    n_needed = next(m for m in ns if power_one_sample(d, m) >= 0.80)
    print(f"  effect size d = {d}: n = {n_needed}")

---
## 6.6 Permutation tests: hypothesis testing without formulas

Everything so far relied on knowing the null distribution analytically. A **permutation
test** builds it empirically, and needs almost no assumptions:

1. Compute the statistic on the real data (e.g. difference in group means)
2. If $H_0$ is true, the group labels are meaningless — so **shuffle** them
3. Recompute the statistic on the shuffled data
4. Repeat 10,000 times: that collection **is** the null distribution
5. The p-value is the fraction of shuffles at least as extreme as the real result

This works for any statistic you can compute — medians, trimmed means, ratios, AUC —
and it makes the logic of hypothesis testing visible rather than abstract.

In [ ]:
# Two teaching methods, small samples, no normality assumption needed
group_a = np.array([72, 68, 81, 75, 79, 66, 88, 73, 77, 70])
group_b = np.array([84, 79, 91, 86, 80, 88, 77, 90, 83, 85])

obs_diff = group_b.mean() - group_a.mean()
print(f"Group A mean : {group_a.mean():.2f}  (n={len(group_a)})")
print(f"Group B mean : {group_b.mean():.2f}  (n={len(group_b)})")
print(f"Observed difference : {obs_diff:.2f} marks")

pooled = np.concatenate([group_a, group_b])
n_a = len(group_a)

B = 20_000
null_diffs = np.empty(B)
for i in range(B):
    shuffled = rng.permutation(pooled)
    null_diffs[i] = shuffled[n_a:].mean() - shuffled[:n_a].mean()

p_perm = (np.abs(null_diffs) >= abs(obs_diff)).mean()
print(f"\nPermutation p-value (two-sided) : {p_perm:.5f}")
print(f"For comparison, Welch t-test     : {stats.ttest_ind(group_b, group_a, equal_var=False).pvalue:.5f}")

In [ ]:
plt.hist(null_diffs, bins=60, color="steelblue", edgecolor="none",
         label="null distribution (20,000 shuffles)")
plt.axvline(obs_diff, color="crimson", lw=2.5, label=f"observed = {obs_diff:.2f}")
plt.axvline(-obs_diff, color="crimson", lw=1, ls=":")
plt.xlabel("difference in group means under random labelling")
plt.ylabel("frequency")
plt.title(f"Permutation test: only {p_perm:.2%} of shuffles are this extreme")
plt.legend(fontsize=8)
plt.show()

print("The histogram IS the null hypothesis, drawn from your own data.")
print("No normality, no equal variances, no formulas -- just relabelling.")

In [ ]:
# The same machinery works for a statistic with no textbook test: the median difference
obs_med = np.median(group_b) - np.median(group_a)
null_med = np.array([
    (lambda s: np.median(s[n_a:]) - np.median(s[:n_a]))(rng.permutation(pooled))
    for _ in range(20_000)
])
print(f"Observed median difference : {obs_med:.2f}")
print(f"Permutation p-value        : {(np.abs(null_med) >= abs(obs_med)).mean():.5f}")

# scipy has this built in
res = stats.permutation_test((group_b, group_a),
                             lambda a, b: np.median(a) - np.median(b),
                             permutation_type="independent", n_resamples=20_000,
                             alternative="two-sided", random_state=1)
print(f"scipy permutation_test     : {res.pvalue:.5f}")

---
## 6.7 Effect size: significance is not importance

A p-value tells you whether an effect is *detectable*. **Effect size** tells you whether it
*matters*. Always report both.

**Cohen's $d$** for a difference in means:

$$d = \frac{\bar{x}_1 - \bar{x}_2}{s_{\text{pooled}}}, \qquad
s_{\text{pooled}} = \sqrt{\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1+n_2-2}}$$

| $\|d\|$ | Conventional label |
|---|---|
| 0.2 | small |
| 0.5 | medium |
| 0.8 | large |

With a big enough $n$, **any** non-zero difference becomes statistically significant. The
demonstration below is the single most important cell in this notebook.

In [ ]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    s_pooled = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na+nb-2))
    return (a.mean() - b.mean()) / s_pooled

print("A genuinely trivial difference (0.1 units on a scale with sd 15):")
print(f"{'n per group':>12} {'p-value':>12} {'Cohen d':>10}  significant?")
for m in (30, 200, 2_000, 20_000, 500_000):
    a = rng.normal(100.0, 15, m)
    b = rng.normal(100.1, 15, m)             # difference of 0.1 -- meaningless in practice
    p = stats.ttest_ind(a, b).pvalue
    print(f"{m:>12,} {p:>12.4f} {cohens_d(a, b):>10.4f}  {'YES' if p < 0.05 else 'no'}")

print("\nThe effect size stays microscopic at every sample size. Only the p-value moves.")
print("Reporting 'p < 0.001' without an effect size is how trivia gets published.")

In [ ]:
# And the reverse: a large, important effect that a small sample cannot detect
print("A large real effect (d = 0.8) measured with too little data:")
print(f"{'n per group':>12} {'p-value':>12} {'Cohen d':>10}  detected?")
for m in (4, 6, 10, 20, 50):
    a = rng.normal(100, 15, m)
    b = rng.normal(112, 15, m)
    p = stats.ttest_ind(a, b).pvalue
    print(f"{m:>12,} {p:>12.4f} {cohens_d(a, b):>10.4f}  {'YES' if p < 0.05 else 'MISSED'}")
print("\n'Not significant' does not mean 'no effect'. It often means 'not enough data'.")

---
## 6.8 Multiple comparisons and p-hacking

Test one hypothesis at $\alpha = 0.05$ and you have a 5% false-positive rate. Test 20
independent hypotheses and the probability of **at least one** false positive is

$$1 - (1 - 0.05)^{20} \approx 0.64$$

This is the **multiple comparisons problem**, and it explains a large share of results that
fail to replicate.

### Corrections

- **Bonferroni:** use $\alpha / m$ for $m$ tests. Simple, conservative, controls the
  family-wise error rate. Costs a lot of power when $m$ is large.
- **Benjamini–Hochberg (FDR):** controls the *expected proportion* of false discoveries
  among rejections. Much more powerful; the standard choice for genomics-scale testing.

In [ ]:
m_tests = 20
print(f"P(at least one false positive in {m_tests} independent tests at alpha=0.05)")
print(f"  = 1 - 0.95^{m_tests} = {1 - 0.95**m_tests:.4f}\n")

for m in (1, 5, 10, 20, 50, 100):
    print(f"  {m:>4} tests -> {1 - 0.95**m:.4f}")

In [ ]:
# Simulate the classic scenario: 20 outcome variables, none with a real effect.
n_outcomes, n_per_group = 20, 40
p_vals = []
for j in range(n_outcomes):
    a = rng.normal(50, 10, n_per_group)
    b = rng.normal(50, 10, n_per_group)          # NO real difference anywhere
    p_vals.append(stats.ttest_ind(a, b).pvalue)
p_vals = np.array(p_vals)

print("p-values from 20 tests where nothing is real:")
print(np.round(np.sort(p_vals), 4))
print(f"\nSignificant at 0.05 (uncorrected)   : {(p_vals < 0.05).sum()} of {n_outcomes}")
print(f"Bonferroni threshold = 0.05/{n_outcomes} = {0.05/n_outcomes:.4f}")
print(f"Significant after Bonferroni        : {(p_vals < 0.05/n_outcomes).sum()}")

# Benjamini-Hochberg by hand
order = np.argsort(p_vals)
ranked = p_vals[order]
bh_thresholds = 0.05 * (np.arange(1, n_outcomes + 1) / n_outcomes)
passed = ranked <= bh_thresholds
k = np.where(passed)[0].max() + 1 if passed.any() else 0
print(f"Significant after Benjamini-Hochberg: {k}")

In [ ]:
# A p-hacking demonstration: keep slicing until something turns up.
base_a = rng.normal(50, 10, 300)
base_b = rng.normal(50, 10, 300)                  # again, no real difference
subgroup = rng.choice(["urban", "rural", "young", "old", "new", "returning"], 300)

print("Overall test:")
print(f"  p = {stats.ttest_ind(base_a, base_b).pvalue:.4f}\n")
print("Now slice by subgroup until something 'works':")
found = []
for g in np.unique(subgroup):
    mask = subgroup == g
    p = stats.ttest_ind(base_a[mask], base_b[mask]).pvalue
    flag = "  <-- 'we found an effect in this segment!'" if p < 0.05 else ""
    print(f"  {g:<11} n={mask.sum():>3}  p = {p:.4f}{flag}")
    if p < 0.05:
        found.append(g)

print()
print("This is p-hacking / HARKing (hypothesising after results are known).")
print("Defences: pre-register your hypotheses, correct for multiplicity, hold out")
print("a confirmation sample, and report every test you ran -- not just the winners.")

---
## 6.9 Choosing the right test

```
What are you comparing?

ONE MEAN against a known value ................. one-sample t-test        (Notebook 7)
TWO INDEPENDENT group means .................... two-sample / Welch t-test (Notebook 7)
TWO PAIRED measurements (before/after) ......... paired t-test            (Notebook 7)
THREE OR MORE group means ...................... one-way ANOVA (F-test)
ONE PROPORTION against a known value ........... binomial / z-test
TWO PROPORTIONS ................................ two-proportion z-test, chi-square
CATEGORICAL association (two variables) ........ chi-square independence  (Notebook 8)
OBSERVED vs EXPECTED category counts ........... chi-square goodness-of-fit (Notebook 8)
CORRELATION between two numeric variables ...... Pearson / Spearman test  (Notebook 5)
VARIANCES between groups ....................... Levene / Bartlett test
NORMALITY of a sample .......................... Shapiro-Wilk, Q-Q plot   (Notebook 3)
ASSUMPTIONS VIOLATED, small n .................. permutation test or a rank test
                                                 (Mann-Whitney, Wilcoxon, Kruskal-Wallis)
```

### Non-parametric alternatives

| Parametric test | Non-parametric equivalent | Use when |
|---|---|---|
| One-sample t | Wilcoxon signed-rank | Skewed, ordinal |
| Two-sample t | Mann–Whitney U | Non-Normal, outliers |
| Paired t | Wilcoxon signed-rank | Non-Normal pairs |
| One-way ANOVA | Kruskal–Wallis | Non-Normal groups |
| Pearson $r$ | Spearman $\rho$ | Monotonic, not linear |

In [ ]:
# Same data, four tests -- see how the choice changes the answer on skewed data
skewed_a = rng.lognormal(3.0, 0.9, 40)
skewed_b = rng.lognormal(3.25, 0.9, 40)

tests = {
    "Student t (equal var)":  stats.ttest_ind(skewed_a, skewed_b, equal_var=True),
    "Welch t (unequal var)":  stats.ttest_ind(skewed_a, skewed_b, equal_var=False),
    "Mann-Whitney U":         stats.mannwhitneyu(skewed_a, skewed_b),
    "t on log-transformed":   stats.ttest_ind(np.log(skewed_a), np.log(skewed_b)),
}
for name, res in tests.items():
    print(f"  {name:<24} statistic = {res.statistic:>10.3f}   p = {res.pvalue:.4f}")

print(f"\nShapiro-Wilk on group A: p = {stats.shapiro(skewed_a).pvalue:.5f} (not Normal)")
print("On skewed data the rank test and the log-transformed t-test are the trustworthy ones.")

---
## 6.10 Reporting results honestly

A complete result has five parts. Compare:

> ❌ *"The treatment was significant (p < 0.05)."*

> ✅ *"The treatment group scored 6.4 points higher on average (95% CI: 2.1 to 10.7),
> a medium effect (Cohen's d = 0.52), t(78) = 2.41, p = 0.018. This exceeds our
> pre-registered threshold of a 5-point improvement."*

The second version reports: **direction**, **magnitude**, **uncertainty**, **effect size**,
and **practical relevance**. The first is unfalsifiable marketing.

In [ ]:
def report(a, b, label_a="group A", label_b="group B", alpha=0.05):
    '''Produce a complete, honest write-up of a two-sample comparison.'''
    res = stats.ttest_ind(a, b, equal_var=False)
    diff = a.mean() - b.mean()
    se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
    dfree = res.df
    t_crit = stats.t(dfree).ppf(1 - alpha/2)
    d = cohens_d(a, b)
    print(f"{label_a}: mean {a.mean():.2f} (sd {a.std(ddof=1):.2f}, n {len(a)})")
    print(f"{label_b}: mean {b.mean():.2f} (sd {b.std(ddof=1):.2f}, n {len(b)})")
    print(f"Difference : {diff:+.2f}  (95% CI {diff - t_crit*se:+.2f} to {diff + t_crit*se:+.2f})")
    print(f"Welch t({dfree:.1f}) = {res.statistic:.3f}, p = {res.pvalue:.4f}")
    print(f"Cohen's d  : {d:+.3f} ({'small' if abs(d)<0.5 else 'medium' if abs(d)<0.8 else 'large'})")
    print(f"Decision   : {'reject H0' if res.pvalue < alpha else 'fail to reject H0'} at alpha={alpha}")

report(rng.normal(78, 12, 40), rng.normal(72, 12, 40), "new curriculum", "old curriculum")

---
## Exercises

**Exercise 1.** A coin is flipped 100 times and lands heads 59 times. Test whether the coin
is fair.
(a) State $H_0$ and $H_1$.
(b) Compute the exact p-value with a binomial test.
(c) Compute the Normal-approximation p-value and compare.
(d) What is the smallest number of heads out of 100 that would be significant at 5%?

In [ ]:
# --- Solution 1 -------------------------------------------------------------
# (a) H0: p = 0.5 (fair)   H1: p != 0.5
heads, flips = 59, 100

exact = stats.binomtest(heads, flips, p=0.5, alternative="two-sided")
print(f"(b) Exact binomial p-value  : {exact.pvalue:.5f}")

p_hat = heads / flips
z = (p_hat - 0.5) / np.sqrt(0.25 / flips)
print(f"(c) Normal approx: z = {z:.4f}, p = {2*stats.norm.sf(abs(z)):.5f}")
print(f"    95% CI for p            : [{exact.proportion_ci().low:.4f}, {exact.proportion_ci().high:.4f}]")

sig = [k for k in range(51, 101)
       if stats.binomtest(k, flips, 0.5, alternative='two-sided').pvalue < 0.05]
print(f"(d) Smallest significant count : {sig[0]} heads out of 100")
print(f"    So 59 heads is {'' if heads >= sig[0] else 'NOT '}enough to reject fairness.")

**Exercise 2.** Design a study. You want to detect a 3-point improvement in a test score
where the population standard deviation is 15.
(a) What is the effect size?
(b) How many participants per group do you need for 80% power at $\alpha = 0.05$?
(c) What power would you have with only 50 per group?
(d) If you could reduce measurement noise to $\sigma = 10$, how does (b) change?

In [ ]:
# --- Solution 2 -------------------------------------------------------------
def n_for_two_sample(d, power=0.80, alpha=0.05):
    za = stats.norm.ppf(1 - alpha/2)
    zb = stats.norm.ppf(power)
    return int(np.ceil(2 * ((za + zb) / d) ** 2))

d1 = 3 / 15
print(f"(a) Cohen's d = 3/15 = {d1:.2f}  (a small effect)")
print(f"(b) n per group for 80% power = {n_for_two_sample(d1):,}")

def power_two_sample(d, n, alpha=0.05):
    za = stats.norm.ppf(1 - alpha/2)
    return stats.norm.sf(za - d*np.sqrt(n/2)) + stats.norm.cdf(-za - d*np.sqrt(n/2))

print(f"(c) Power with n = 50 per group = {power_two_sample(d1, 50):.3f}  (badly underpowered)")

d2 = 3 / 10
print(f"(d) With sigma = 10: d = {d2:.2f}, n per group = {n_for_two_sample(d2):,}")
print("    Reducing noise is often cheaper than recruiting more participants.")

**Exercise 3.** Write a permutation test for the difference in the **90th percentile**
between two groups (there is no standard formula for this). Apply it to two samples of
response times and report the p-value.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
control  = rng.lognormal(5.0, 0.6, 120)     # response times, ms
variant  = rng.lognormal(5.0, 0.75, 120)    # same median, fatter tail

def p90_diff(a, b):
    return np.percentile(a, 90) - np.percentile(b, 90)

obs = p90_diff(variant, control)
pool = np.concatenate([variant, control])
k = len(variant)

null = np.array([
    (lambda s: p90_diff(s[:k], s[k:]))(rng.permutation(pool)) for _ in range(20_000)
])
p_val = (np.abs(null) >= abs(obs)).mean()

print(f"p90 control : {np.percentile(control, 90):.1f} ms")
print(f"p90 variant : {np.percentile(variant, 90):.1f} ms")
print(f"Observed difference : {obs:+.1f} ms")
print(f"Permutation p-value : {p_val:.5f}")
print(f"\nFor contrast, the MEDIANS barely differ:")
print(f"  medians {np.median(control):.1f} vs {np.median(variant):.1f}, "
      f"Mann-Whitney p = {stats.mannwhitneyu(control, variant).pvalue:.3f}")
print("A test on the tail can detect a regression that a test on the centre misses.")

plt.hist(null, bins=60, color="steelblue", edgecolor="none")
plt.axvline(obs, color="crimson", lw=2.5, label=f"observed {obs:+.1f}")
plt.title(f"Permutation null for the p90 difference (p = {p_val:.4f})")
plt.legend(); plt.show()

**Exercise 4 (challenge).** Your team runs an A/B test with 12 metrics and reports the two
that came out significant.
(a) If none of the 12 metrics is genuinely affected, what is the chance at least one looks
significant?
(b) Simulate 5,000 such experiments and record how often the *uncorrected*, *Bonferroni*
and *Benjamini–Hochberg* procedures produce at least one false discovery.
(c) Now add three metrics with a real effect and compare how many true effects each
procedure detects.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
def bh_reject(pvals, q=0.05):
    '''Benjamini-Hochberg: returns a boolean mask of rejections.'''
    pv = np.asarray(pvals); m = len(pv)
    order = np.argsort(pv)
    thresh = q * (np.arange(1, m + 1) / m)
    passed = pv[order] <= thresh
    mask = np.zeros(m, dtype=bool)
    if passed.any():
        cutoff = np.where(passed)[0].max()
        mask[order[:cutoff + 1]] = True
    return mask

n_metrics, n_grp, sims = 12, 60, 2_000
print(f"(a) P(at least one of 12 significant | nothing real) = {1 - 0.95**12:.4f}\n")

def run(effects):
    counts = {"uncorrected": 0, "bonferroni": 0, "BH": 0}
    detected = {"uncorrected": 0, "bonferroni": 0, "BH": 0}
    real = np.array(effects) != 0
    for _ in range(sims):
        pv = np.array([stats.ttest_ind(rng.normal(0, 1, n_grp),
                                       rng.normal(e, 1, n_grp)).pvalue for e in effects])
        r_unc = pv < 0.05
        r_bon = pv < 0.05 / n_metrics
        r_bh  = bh_reject(pv)
        for name, r in [("uncorrected", r_unc), ("bonferroni", r_bon), ("BH", r_bh)]:
            counts[name]   += bool((r & ~real).any())     # any FALSE discovery
            detected[name] += int((r & real).sum())        # true discoveries
    return counts, detected

# (b) nothing real
c0, _ = run([0.0] * n_metrics)
print("(b) With NO real effects -- proportion of experiments with >=1 false positive:")
for k_, v in c0.items():
    print(f"    {k_:<12} {v/sims:.4f}")

# (c) three real effects of size d = 0.6
effects = [0.6, 0.6, 0.6] + [0.0] * (n_metrics - 3)
c1, d1_ = run(effects)
print("\n(c) With 3 real effects (d = 0.6):")
print(f"    {'procedure':<12} {'>=1 false pos':>14} {'true effects found (of 3)':>28}")
for k_ in c0:
    print(f"    {k_:<12} {c1[k_]/sims:>14.4f} {d1_[k_]/sims:>28.2f}")
print("\nBonferroni controls false positives hardest but finds the fewest real effects.")
print("BH sits in between and is usually the right default when you have many metrics.")
print("Uncorrected testing is only defensible for a single pre-registered primary metric.")

---
## Summary

| Concept | Key point |
|---|---|
| $H_0$ / $H_1$ | $H_0$ carries the equality; state both before seeing data |
| Test statistic | (observed − expected) / standard error |
| p-value | $P(\text{data this extreme} \mid H_0)$ — **not** $P(H_0)$ |
| $\alpha$ | Type I (false positive) rate you accept, usually 0.05 |
| $\beta$ / power | Miss rate / detection rate; target power ≥ 0.80 |
| Trade-off | Lower $\alpha$ → higher $\beta$; only more data improves both |
| One vs two-tailed | One-tailed only when pre-registered and directionally justified |
| Permutation test | Shuffle labels to build the null; almost assumption-free |
| Effect size | Cohen's $d$; always report alongside $p$ |
| Multiple testing | 20 tests → 64% chance of a false positive; use Bonferroni or BH |
| Honest reporting | Direction + magnitude + CI + effect size + practical relevance |

**Next up:** [Notebook 7 — t-Tests](7.%20t-Test.ipynb), the most-used hypothesis test in
practice, in all three of its forms.